In [ ]:
import pandas as pd
import polars as pl
import numpy as np
from pathlib import Path

from pip._internal.operations.install import wheel
from polars import selectors as cs

schedule = Path('data/schedule')
statcast = Path('data/statcast')

In [ ]:
sched_schema = pl.Schema(
    {
        'date': pl.Date,
        'total_items': pl.UInt8,
        'total_events': pl.UInt8,
        'total_games': pl.UInt8,
        'total_games_in_progress': pl.UInt8,
        'game_pk': pl.UInt32,
        'game_guid': pl.String,
        'link': pl.String,
        'game_type': pl.Categorical(),
        'season': pl.UInt16,
        'game_date': pl.Datetime(),
        'official_date': pl.Date,
        'is_tie': pl.Boolean,
        'game_number': pl.UInt8,
        'public_facing': pl.Boolean,
        'double_header': pl.Categorical(),
        'gameday_type': pl.Categorical(),
        'tiebreaker': pl.Boolean,
        'calendar_event_id': pl.String,
        'season_display': pl.UInt16,
        'day_night': pl.Categorical(),
        'description': pl.String,
        'scheduled_innings': pl.UInt8,
        'reverse_home_away_status': pl.Boolean,
        'inning_break_length': pl.UInt16,
        'games_in_series': pl.UInt8,
        'series_game_number': pl.UInt8,
        'series_description': pl.String,
        'record_source': pl.String,
        'if_necessary': pl.Boolean,
        'if_necessary_description': pl.String,
        'status_abstract_game_state': pl.Categorical(),
        'status_coded_game_state': pl.Categorical(),
        'status_detailed_state': pl.Categorical(),
        'status_status_code': pl.Categorical(),
        'status_start_time_tbd': pl.Boolean,
        'status_abstract_game_code': pl.Categorical(),
        'teams_away_score': pl.UInt8,
        'teams_away_is_winner': pl.Boolean,
        'teams_away_split_squad': pl.Boolean,
        'teams_away_series_number': pl.UInt8,
        'teams_away_team_id': pl.UInt16,
        'teams_away_team_name': pl.String,
        'teams_away_team_link': pl.String,
        'teams_away_league_record_wins': pl.UInt16,
        'teams_away_league_record_losses': pl.UInt16,
        'teams_away_league_record_ties': pl.UInt16,
        'teams_away_league_record_pct': pl.Float64,
        'teams_home_score': pl.UInt8,
        'teams_home_is_winner': pl.Boolean,
        'teams_home_split_squad': pl.Boolean,
        'teams_home_series_number': pl.UInt8,
        'teams_home_team_id': pl.UInt16,
        'teams_home_team_name': pl.String,
        'teams_home_team_link': pl.String,
        'teams_home_league_record_wins': pl.UInt16,
        'teams_home_league_record_losses': pl.UInt16,
        'teams_home_league_record_ties': pl.UInt16,
        'teams_home_league_record_pct': pl.Float64,
        'venue_id': pl.UInt32,
        'venue_name': pl.String,
        'venue_link': pl.String,
        'content_link': pl.String,
        'status_reason': pl.Categorical(),
        'reschedule_date': pl.Datetime(),
        'reschedule_game_date': pl.Date,
        'resume_date': pl.Datetime(),
        'resume_game_date': pl.Date,
        'resumed_from': pl.Datetime(),
        'resumed_from_date': pl.Date,
        'rescheduled_from': pl.Datetime(),
        'rescheduled_from_date': pl.Date,
        'events': pl.String
    }
)

In [ ]:
sc_schema = pl.Schema(
    {
        'pitch_type': pl.Categorical(),
        'game_date': pl.Date,
        'release_speed': pl.Float64,
        'release_pos_x': pl.Float64,
        'release_pos_z': pl.Float64,
        'player_name': pl.String,
        'batter': pl.UInt32,
        'pitcher': pl.UInt32,
        'events': pl.Categorical(),
        'description': pl.Categorical(),
        'spin_dir': pl.UInt16,
        'spin_rate_deprecated': pl.UInt16,
        'break_angle_deprecated': pl.UInt16,
        'break_length_deprecated': pl.Float64,
        'zone': pl.Categorical(),
        'des': pl.String,
        'game_type': pl.Categorical(),
        'stand': pl.Categorical(),
        'p_throws': pl.Categorical(),
        'home_team': pl.Categorical(),
        'away_team': pl.Categorical(),
        'type': pl.Categorical(),
        'hit_location': pl.Categorical(),
        'bb_type': pl.Categorical(),
        'balls': pl.UInt8,
        'strikes': pl.UInt8,
        'game_year': pl.UInt16,
        'pfx_x': pl.Float64,
        'pfx_z': pl.Float64,
        'plate_x': pl.Float64,
        'plate_z': pl.Float64,
        'on_3b': pl.UInt32,
        'on_2b': pl.UInt32,
        'on_1b': pl.UInt32,
        'outs_when_up': pl.UInt8,
        'inning': pl.UInt8,
        'inning_topbot': pl.Categorical(),
        'hc_x': pl.Float64,
        'hc_y': pl.Float64,
        'tfs_deprecated': pl.String,
        'tfs_zulu_deprecated': pl.String,
        'umpire': pl.UInt32,
        'sv_id': pl.String,
        'vx0': pl.Float64,
        'vy0': pl.Float64,
        'vz0': pl.Float64,
        'ax': pl.Float64,
        'ay': pl.Float64,
        'az': pl.Float64,
        'sz_top': pl.Float64,
        'sz_bot': pl.Float64,
        'hit_distance_sc': pl.UInt16,
        'launch_speed': pl.Float64,
        'launch_angle': pl.Int16,
        'effective_speed': pl.Float64,
        'release_spin_rate': pl.UInt16,
        'release_extension': pl.Float64,
        'game_pk': pl.UInt32,
        'fielder_2': pl.UInt32,
        'fielder_3': pl.UInt32,
        'fielder_4': pl.UInt32,
        'fielder_5': pl.UInt32,
        'fielder_6': pl.UInt32,
        'fielder_7': pl.UInt32,
        'fielder_8': pl.UInt32,
        'fielder_9': pl.UInt32,
        'release_pos_y': pl.Float64,
        'estimated_ba_using_speedangle': pl.Float64,
        'estimated_woba_using_speedangle': pl.Float64,
        'woba_value': pl.Float64,
        'woba_denom': pl.UInt8,
        'babip_value': pl.UInt8,
        'iso_value': pl.UInt8,
        'launch_speed_angle': pl.UInt8,
        'at_bat_number': pl.UInt16,
        'pitch_number': pl.UInt8,
        'pitch_name': pl.Categorical(),
        'home_score': pl.UInt8,
        'away_score': pl.UInt8,
        'bat_score': pl.UInt8,
        'fld_score': pl.UInt8,
        'post_away_score': pl.UInt8,
        'post_home_score': pl.UInt8,
        'post_bat_score': pl.UInt8,
        'post_fld_score': pl.UInt8,
        'if_fielding_alignment': pl.Categorical(),
        'of_fielding_alignment': pl.Categorical(),
        'spin_axis': pl.UInt16,
        'delta_home_win_exp': pl.Float64,
        'delta_run_exp': pl.Float64,
        'bat_speed': pl.Float64,
        'swing_length': pl.Float64,
        'estimated_slg_using_speedangle': pl.Float64,
        'delta_pitcher_run_exp': pl.Float64,
        'hyper_speed': pl.Float64,
        'home_score_diff': pl.Int8,
        'bat_score_diff': pl.Int8,
        'home_win_exp': pl.Float64,
        'bat_win_exp': pl.Float64,
        'age_pit_legacy': pl.UInt8,
        'age_bat_legacy': pl.UInt8,
        'age_pit': pl.UInt8,
        'age_bat': pl.UInt8,
        'n_thruorder_pitcher': pl.UInt8,
        'n_priorpa_thisgame_player_at_bat': pl.UInt8,
        'pitcher_days_since_prev_game': pl.UInt8,
        'batter_days_since_prev_game': pl.UInt8,
        'pitcher_days_until_next_game': pl.UInt8,
        'batter_days_until_next_game': pl.UInt8,
        'api_break_z_with_gravity': pl.Float64,
        'api_break_x_arm': pl.Float64,
        'api_break_x_batter_in': pl.Float64,
        'arm_angle': pl.Float64,
        'attack_angle': pl.Float64,
        'attack_direction': pl.Float64,
        'swing_path_tilt': pl.Float64,
        'intercept_ball_minus_batter_pos_x_inches': pl.Float64,
        'intercept_ball_minus_batter_pos_y_inches': pl.Float64,
    }
)

In [ ]:
sc_schema_clean = pl.Schema(
    {
        'pitch_type': pl.Categorical(),
        'game_date': pl.Date,
        'release_speed': pl.Float64,
        'release_pos_x': pl.Float64,
        'release_pos_z': pl.Float64,
        'player_name': pl.String,
        'batter': pl.UInt32,
        'pitcher': pl.UInt32,
        'events': pl.Categorical(),
        'description': pl.Categorical(),
        'zone': pl.Categorical(),
        'des': pl.String,
        'game_type': pl.Categorical(),
        'stand': pl.Categorical(),
        'p_throws': pl.Categorical(),
        'home_team': pl.Categorical(),
        'away_team': pl.Categorical(),
        'type': pl.Categorical(),
        'hit_location': pl.Categorical(),
        'bb_type': pl.Categorical(),
        'balls': pl.UInt8,
        'strikes': pl.UInt8,
        'game_year': pl.UInt16,
        'pfx_x': pl.Float64,
        'pfx_z': pl.Float64,
        'plate_x': pl.Float64,
        'plate_z': pl.Float64,
        'on_3b': pl.UInt32,
        'on_2b': pl.UInt32,
        'on_1b': pl.UInt32,
        'outs_when_up': pl.UInt8,
        'inning': pl.UInt8,
        'inning_topbot': pl.Categorical(),
        'hc_x': pl.Float64,
        'hc_y': pl.Float64,
        'vx0': pl.Float64,
        'vy0': pl.Float64,
        'vz0': pl.Float64,
        'ax': pl.Float64,
        'ay': pl.Float64,
        'az': pl.Float64,
        'sz_top': pl.Float64,
        'sz_bot': pl.Float64,
        'hit_distance_sc': pl.UInt16,
        'launch_speed': pl.Float64,
        'launch_angle': pl.Int16,
        'effective_speed': pl.Float64,
        'release_spin_rate': pl.UInt16,
        'release_extension': pl.Float64,
        'game_pk': pl.UInt32,
        'fielder_2': pl.UInt32,
        'fielder_3': pl.UInt32,
        'fielder_4': pl.UInt32,
        'fielder_5': pl.UInt32,
        'fielder_6': pl.UInt32,
        'fielder_7': pl.UInt32,
        'fielder_8': pl.UInt32,
        'fielder_9': pl.UInt32,
        'release_pos_y': pl.Float64,
        'estimated_ba_using_speedangle': pl.Float64,
        'estimated_woba_using_speedangle': pl.Float64,
        'woba_value': pl.Float64,
        'woba_denom': pl.UInt8,
        'babip_value': pl.UInt8,
        'iso_value': pl.UInt8,
        'launch_speed_angle': pl.UInt8,
        'at_bat_number': pl.UInt16,
        'pitch_number': pl.UInt8,
        'pitch_name': pl.Categorical(),
        'home_score': pl.UInt8,
        'away_score': pl.UInt8,
        'bat_score': pl.UInt8,
        'fld_score': pl.UInt8,
        'post_away_score': pl.UInt8,
        'post_home_score': pl.UInt8,
        'post_bat_score': pl.UInt8,
        'post_fld_score': pl.UInt8,
        'if_fielding_alignment': pl.Categorical(),
        'of_fielding_alignment': pl.Categorical(),
        'spin_axis': pl.UInt16,
        'delta_home_win_exp': pl.Float64,
        'delta_run_exp': pl.Float64,
        'bat_speed': pl.Float64,
        'swing_length': pl.Float64,
        'estimated_slg_using_speedangle': pl.Float64,
        'delta_pitcher_run_exp': pl.Float64,
        'hyper_speed': pl.Float64,
        'home_score_diff': pl.Int8,
        'bat_score_diff': pl.Int8,
        'home_win_exp': pl.Float64,
        'bat_win_exp': pl.Float64,
        'age_pit_legacy': pl.UInt8,
        'age_bat_legacy': pl.UInt8,
        'age_pit': pl.UInt8,
        'age_bat': pl.UInt8,
        'n_thruorder_pitcher': pl.UInt8,
        'n_priorpa_thisgame_player_at_bat': pl.UInt8,
        'pitcher_days_since_prev_game': pl.UInt8,
        'batter_days_since_prev_game': pl.UInt8,
        'pitcher_days_until_next_game': pl.UInt8,
        'batter_days_until_next_game': pl.UInt8,
        'api_break_z_with_gravity': pl.Float64,
        'api_break_x_arm': pl.Float64,
        'api_break_x_batter_in': pl.Float64,
        'arm_angle': pl.Float64,
        'attack_angle': pl.Float64,
        'attack_direction': pl.Float64,
        'swing_path_tilt': pl.Float64,
        'intercept_ball_minus_batter_pos_x_inches': pl.Float64,
        'intercept_ball_minus_batter_pos_y_inches': pl.Float64,
    }
)


In [ ]:
def clean_schedules(schema):
    for f in schedule.glob('schedule_????.csv'):
        df = pd.read_csv(f)
        map_dict = {'N': False, 'Y': True}

        df['tiebreaker'] = df['tiebreaker'].map(map_dict)

        df['if_necessary'] = df['if_necessary'].map(map_dict)
        for col, dtype in zip(sched_schema.names(), sched_schema.dtypes()):
            if col not in df.columns:
                df[col] = np.nan

            if dtype == pl.UInt8:
                df[col] = df[col].astype('Int64')
            elif dtype == pl.UInt16:
                df[col] = df[col].astype('Int64')
            elif dtype == pl.UInt32:
                df[col] = df[col].astype('Int64')
            elif dtype == pl.UInt64:
                df[col] = df[col].astype('Int64')
        df[schema.names()].to_csv(schedule/('clean_'+f.name), index=False)

        df = pl.read_csv(schedule/('clean_'+f.name))
        df = df.filter(
            ~pl.col('is_tie').is_null()
        ).with_columns(
            pl.col('description').fill_null('no description'),
            pl.col('inning_break_length').fill_null(strategy='backward'),
            pl.col('games_in_series').fill_null(strategy='zero'),
            pl.col('series_game_number').fill_null(strategy='zero'),
            pl.col('teams_away_is_winner').fill_null(False),
            pl.col('teams_home_is_winner').fill_null(False),
            pl.col('teams_away_series_number').fill_null(strategy='zero'),
            pl.col('teams_home_series_number').fill_null(strategy='zero'),
        )
        df.write_csv(schedule/('clean_'+f.name))
clean_schedules(sched_schema)

In [ ]:
def clean_statcast(schema):
    for f in statcast.glob('statcast_????.csv'):
        df = pl.read_csv(f, schema = schema)
        df = df.with_columns(
            pl.col('events').fill_null(pl.col('description')),
            pl.col('des').fill_null(pl.col('description')),
            pl.when(
                pl.col('hit_location').is_null() &
                (pl.col('events') == 'home_run')
            ).then(
                pl.col('hit_location').fill_null("HR"),
            ).when(
                pl.col('hit_location').is_null() &
                ((pl.col('events') == 'double') & pl.col('des').str.contains('ground-rule double'))
            ).then(
                pl.col('hit_location').fill_null("GRD"),
            ).when(
                pl.col('hit_location').is_null() &
                pl.col('des').str.contains('fan interference') &
                ~pl.col('des').str.contains('ground-rule double')
            ).then(
                pl.col('hit_location').fill_null("FAN"),
            ).when(
                pl.col('hit_location').is_null() &
                (pl.col('type') == 'X')
            ).then(
                pl.col('hit_location').fill_null("UNKNOWN"),
            ).otherwise(
                pl.col('hit_location').fill_null("NO_HIT")
            ),
            pl.col('bb_type').fill_null("not_in_play"),
            cs.matches('on_[1-3]b').fill_null(strategy='zero'),
        ).select(
            pl.exclude(
                'spin_dir',
                'spin_rate_deprecated',
                'break_angle_deprecated',
                'break_length_deprecated',
                'tfs_zulu_deprecated',
                'tfs_deprecated',
                'umpire',
                'sv_id'
            )
        )

        df.write_csv(statcast/('clean_'+f.name))
clean_statcast(sc_schema)

In [ ]:
df_sch = pl.scan_csv(schedule/'clean_schedule_????.csv', schema = sched_schema)

df_sc = pl.scan_csv(statcast/'clean_statcast_????.csv', schema = sc_schema_clean)


In [ ]:
df_sch.collect().null_count()

In [ ]:
df_sc.describe()

In [ ]:
df_sc.filter(
    ~pl.col('hit_location').is_null() &
    (pl.col('type') != 'X') &
    (pl.col('hit_location') == "2") &
    (pl.col('events') != 'strikeout') &
    (pl.col('bb_type') != 'not_in_play')
).select(
    'des',
    'events',
    'description',
    'type',
    'hit_location',
    'bb_type'
).collect()

In [ ]:
df_sc.filter(
    pl.col('hit_location').is_null()
).group_by('type').len().collect()

In [ ]:
df_sc.select(
    cs.matches('hc_[xy]'),
    'hit_distance_sc'
).drop_nulls().collect().sample(1000)


In [ ]:
df_sc.with_columns(
    (np.atan(
        (pl.col('hc_x')-125.42)/(198.27 - pl.col('hc_y'))
    )* 180 /np.pi *.75).alias('spray_angle')
).drop_nulls().select(
    cs.matches('hc_[xy]'),
    'spray_angle'
).describe()

In [ ]:
df_sc.filter(
    pl.col('hc_x').is_null().xor(pl.col('hit_distance_sc').is_null()) &
    (pl.col('description') != 'foul')
).select(
    'des',

    'description',
    cs.matches('hc_[xy]'),
    'hit_distance_sc'
).collect().sample(1000)